# Delay by train type, September 2026

Only `data-2026-09.parquet` is used. Canceled stops, bus and rail-replacement rows are dropped, and types with few stops are left out of the charts.

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

DATA = "../data/monthly_processed_data/data-2026-09.parquet"
NON_TRAIN = ["Bus", "bus", "SEV", "Bsv", "BSv"]
MIN_STOPS = 20_000  # types below this are too small to read a mean from

# validated categorical slots 1-4 (blue, orange, aqua, yellow) and neutral inks
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e3e2de", "#fcfcfb"

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": GRID,
        "axes.labelcolor": INK_2,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "text.color": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "font.size": 10,
    }
)

## Load and clean

In [ ]:
raw = pl.scan_parquet(DATA)
n_all = raw.select(pl.len()).collect().item()

clean = raw.filter(
    ~pl.col("arrival_is_canceled"),
    ~pl.col("train_type").is_in(NON_TRAIN),
    pl.col("train_type").is_not_null(),
    pl.col("delay_in_min").is_not_null(),
)
n_clean = clean.select(pl.len()).collect().item()
print(
    f"{n_all:,} stop events -> {n_clean:,} after cleaning ({n_all - n_clean:,} dropped)"
)

## Summary per train type

In [ ]:
d = pl.col("delay_in_min")
by_type = (
    clean.group_by("train_type")
    .agg(
        pl.len().alias("stops"),
        d.mean().alias("mean"),
        d.median().alias("median"),
        d.quantile(0.9).alias("p90"),
        (d >= 6).mean().alias("share_ge6"),
    )
    .filter(pl.col("stops") >= MIN_STOPS)
    .sort("mean", descending=True)
    .collect()
)
overall_mean = clean.select(d.mean()).collect().item()
print(
    f"{by_type.height} types with >= {MIN_STOPS:,} stops, overall mean {overall_mean:.2f} min"
)
by_type

## Mean delay per type

Bars are sorted by mean delay. The dashed line is the mean over all trains.

In [ ]:
rows = by_type.sort("mean").to_dicts()
fig, ax = plt.subplots(figsize=(8, 0.28 * len(rows) + 1.2))
ax.barh(
    [r["train_type"] for r in rows], [r["mean"] for r in rows], color=BLUE, height=0.6
)
ax.axvline(overall_mean, color=INK_2, linestyle="--", linewidth=1)
ax.text(
    overall_mean,
    len(rows) - 0.3,
    f" all trains {overall_mean:.1f}",
    color=INK_2,
    va="bottom",
)
for i, r in enumerate(rows):
    ax.text(
        r["mean"] + 0.15,
        i,
        f"{r['mean']:.1f}",
        va="center",
        color=INK_2,
        fontsize=9,
        bbox={"fc": SURFACE, "ec": "none", "pad": 0.5},
    )
ax.grid(axis="y", visible=False)
ax.set_xlabel("Mean arrival delay (minutes)")
ax.set_title("Mean delay by train type, September 2026", loc="left", color=INK)
plt.show()

## Typical vs bad days

The bar runs from the median to the 90th percentile and the dot is the mean. A mean far to the right of the median means a long tail of big delays.

In [ ]:
rows = by_type.sort("p90").to_dicts()
fig, ax = plt.subplots(figsize=(8, 0.28 * len(rows) + 1.2))
for i, r in enumerate(rows):
    ax.plot(
        [r["median"], r["p90"]],
        [i, i],
        color=BLUE,
        linewidth=3,
        solid_capstyle="round",
        alpha=0.45,
    )
    ax.plot(r["median"], i, "o", color=BLUE, markersize=5, markeredgecolor=SURFACE)
    ax.plot(r["mean"], i, "o", color=ORANGE, markersize=7, markeredgecolor=SURFACE)
ax.set_yticks(range(len(rows)), [r["train_type"] for r in rows])
ax.grid(axis="y", visible=False)
ax.set_xlabel("Arrival delay (minutes)")
ax.set_title("Median to p90 range and mean, by train type", loc="left", color=INK)
ax.plot([], [], "o", color=BLUE, label="median")
ax.plot([], [], "o", color=ORANGE, label="mean")
ax.legend(frameon=False, loc="lower right")
plt.show()

## Day by day

Daily mean delay for four large types (long distance, regional and S-Bahn).

In [ ]:
TYPES = {"ICE": BLUE, "IC": ORANGE, "RE": AQUA, "S": YELLOW}
daily = (
    clean.filter(pl.col("train_type").is_in(list(TYPES)))
    .group_by(pl.col("time").dt.date().alias("day"), "train_type")
    .agg(d.mean().alias("mean"))
    .sort("day")
    .collect()
)
fig, ax = plt.subplots(figsize=(9, 4))
for t, color in TYPES.items():
    s = daily.filter(pl.col("train_type") == t)
    ax.plot(s["day"], s["mean"], color=color, linewidth=2, label=t)
    ax.text(s["day"][-1], s["mean"][-1], f"  {t}", color=INK, va="center")
ax.set_ylabel("Mean arrival delay (minutes)")
ax.set_title("Daily mean delay, September 2026", loc="left", color=INK)
ax.legend(frameon=False, ncols=4, loc="upper left")
fig.autofmt_xdate()
plt.show()